In [1]:
from __future__ import annotations

from dataclasses import dataclass
import glob
import xarray as xr
import numpy as np
from pathlib import Path

from typing import Dict, Optional, Tuple, List, Any

import numpy as np
import xarray as xr
import pandas as pd
import datetime as _dt

from ExperimentUtil import (
    run_dict,
    open_var_from_files,
    get_comp_dir,
    find_nc_files,
    collect_variable_list,
    flatten_variable_dict,
    open_compvar,
    set_monthly_time_to_midmonth_cftime
)

In [2]:
# ============================================================
# Config
# ============================================================
@dataclass(frozen=True)
class RmmProjectConfig:
    # EOF file (must contain EOF_OLR/EOF_U850/EOF_U200, norm_std_*, pc_std_mode, rmm_scale, lat/lon coords)
    eof_nc: str

    # Obs inputs used ONLY to build DOY climatology (cacheable)
    noaa_olr_file: str
    era5_u850_file: str
    era5_u200_file: str

    v_olr: str = "FLUT"
    v_u850: str = "U850"
    v_u200: str = "U200"

    time_dim: str = "time"
    lat_dim: str = "lat"
    lon_dim: str = "lon"
    init_dim: str = "init"

    # Domain must match EOF training
    lat_range: Tuple[float, float] = (-15.0, 15.0)
    lon_range: Tuple[float, float] = (0.0, 360.0)

    # Climatology training window (fallback if EOF attrs absent)
    climo_train_start: str = "1986-01-01"
    climo_train_end: str = "2017-12-31"

    drop_leap_day: bool = True
    doy_smooth_days: int = 30

    cache_dir: str = "./rmm_cache"
    use_cache: bool = True
    overwrite_cache: bool = False

    interp_method: str = "linear"

    # Optional: apply to model projection time trimming
    verify_start: Optional[str] = None
    verify_end: Optional[str] = None
    
    model_time_shift_hours: int = -12   
    obs_time_shift_hours: int = 0

# ============================================================
# Result container
# ============================================================
@dataclass
class RmmProjectionResult:
    fc_rmm_native: xr.Dataset               # (init,time) typically
    fc_rmm_timeseries: Optional[xr.Dataset] # (time) if single-init else None

def report_missing_days(time_vals: np.ndarray, tag: str = "", *, drop_feb29: bool = True) -> None:
    t = np.asarray(time_vals).astype("datetime64[D]")
    if t.size < 2:
        return
    t = np.sort(t)

    full = np.arange(
        t[0],
        t[-1] + np.timedelta64(1, "D"),
        np.timedelta64(1, "D"),
        dtype="datetime64[D]",
    )

    if drop_feb29:
        # Remove Feb 29 from the expected "full" timeline
        # Works for datetime64[D] by converting to pandas for month/day extraction.
        import pandas as pd
        s = pd.to_datetime(full.astype("datetime64[ns]"))
        full = full[~((s.month == 2) & (s.day == 29))]

    missing = np.setdiff1d(full, np.unique(t))
    if missing.size:
        print(f"[{tag}] MISSING days: {missing.size} (first 20) {missing[:20]}")
    else:
        print(f"[{tag}] Missing days: 0")

def diag_time_axis(
    da: xr.DataArray,
    time_dim: str,
    tag: str,
    max_show_missing: int = 20,
    *,
    drop_feb29_expected: bool = False,   # set True for "noleap expectations" on datetime64 timelines
) -> None:
    if time_dim not in da.coords:
        print(f"[{tag}] no coord '{time_dim}'")
        return

    t = da[time_dim].values
    if t.size == 0:
        print(f"[{tag}] empty time")
        return

    # ---- Try numpy datetime64 path ----
    try:
        tns = t.astype("datetime64[ns]")
        td = tns.astype("datetime64[D]")
        print(f"[{tag}] time(ns): {tns[0]} → {tns[-1]}  n={tns.size}")

        if td.size > 1:
            ti = td.astype("int64")
            dt = np.unique(np.diff(ti))
            nunq = np.unique(ti).size
            print(f"[{tag}] time(D):  {td[0]} → {td[-1]}  n={td.size}  unique_days={nunq}")
            print(f"[{tag}] Δday(unique)={dt[:10]}{'...' if dt.size>10 else ''}")

            full = np.arange(
                td.min(),
                td.max() + np.timedelta64(1, "D"),
                np.timedelta64(1, "D"),
                dtype="datetime64[D]",
            )

            if drop_feb29_expected:
                # remove Feb 29 from expected range
                import pandas as pd
                s = pd.to_datetime(full.astype("datetime64[ns]"))
                full = full[~((s.month == 2) & (s.day == 29))]

            missing = np.setdiff1d(full, np.unique(td))
            if missing.size:
                print(f"[{tag}] MISSING days: {missing.size} (first {max_show_missing}) {missing[:max_show_missing]}")
            else:
                print(f"[{tag}] Missing days: 0")
        return

    except Exception:
        pass  # fall through to cftime branch

    # ---- cftime-safe branch ----
    # t is typically dtype=object with cftime.datetime objects
    t0, t1 = t[0], t[-1]
    print(f"[{tag}] time(cftime): {t0} → {t1}  n={t.size}")

    if t.size == 1:
        return

    # day deltas
    deltas = np.array([(t[i] - t[i - 1]).days for i in range(1, t.size)], dtype=int)
    dt_unique = np.unique(deltas)
    print(f"[{tag}] Δday(unique)={dt_unique[:10]}{'...' if dt_unique.size>10 else ''}")

    # enumerate missing dates for the first few gaps
    missing = []
    import datetime as _dt
    one = _dt.timedelta(days=1)

    for i in range(1, t.size):
        dd = (t[i] - t[i - 1]).days
        if dd > 1:
            cur = t[i - 1]
            for _ in range(dd - 1):
                cur = cur + one
                missing.append(cur)
                if len(missing) >= max_show_missing:
                    break
        if len(missing) >= max_show_missing:
            break

    if missing:
        print(f"[{tag}] MISSING days: (showing up to {max_show_missing}) {missing}")
    else:
        print(f"[{tag}] Missing days: 0")


def standardize_spacetime(
    da: xr.DataArray,
    *,
    time_dim: str = "time",
    lat_dim: str = "lat",
    lon_dim: str = "lon",
    rename_latlon: bool = True,
    # NEW:
    time_shift_hours: int = 0,          # e.g. -1 for “labeled at end of day”
    floor_time_to_day: bool = True,
    drop_duplicate_time: bool = True,
    lon_to_0_360: bool = True,
    drop_lon_360: bool = True,
    sort_coords: bool = True,
    ensure_lat_ascending: bool = True,
    cast_dtype: Optional[str] = None,
) -> xr.DataArray:
    # ---- rename common coord names
    if rename_latlon:
        rename_map = {}
        if "latitude" in da.dims and lat_dim not in da.dims:
            rename_map["latitude"] = lat_dim
        if "longitude" in da.dims and lon_dim not in da.dims:
            rename_map["longitude"] = lon_dim
        if rename_map:
            da = da.rename(rename_map)

    # ---- NEW: shift time (model-only usually)
    if (time_dim in da.coords) and (time_shift_hours != 0):
        t = da[time_dim].values.astype("datetime64[ns]")
        da = da.assign_coords({time_dim: t + np.timedelta64(int(time_shift_hours), "h")})

    # ---- time: floor to day (00Z), remove duplicates
    if (time_dim in da.coords) and floor_time_to_day:
        t00 = np.array(da[time_dim].values, dtype="datetime64[D]").astype("datetime64[ns]")
        da = da.assign_coords({time_dim: t00})

        if sort_coords and (time_dim in da.dims):
            da = da.sortby(time_dim)

        if drop_duplicate_time and (time_dim in da.dims):
            t = da[time_dim].values
            _, idx = np.unique(t, return_index=True)
            if idx.size != t.size:
                da = da.isel({time_dim: np.sort(idx)})

    # ---- lon: force [0,360) and sort
    if lon_dim in da.coords and lon_to_0_360:
        lon = da[lon_dim]
        da = da.assign_coords({lon_dim: (lon % 360).astype(lon.dtype)})

        if drop_lon_360:
            da = da.where(da[lon_dim] < 360.0, drop=True)

        if sort_coords:
            da = da.sortby(lon_dim)

        # optional: de-dup lon after wrapping
        if lon_dim in da.dims:
            lonv = da[lon_dim].values
            _, idx = np.unique(lonv, return_index=True)
            if idx.size != lonv.size:
                da = da.isel({lon_dim: np.sort(idx)})

    # ---- lat: ascending and sorted
    if lat_dim in da.coords and ensure_lat_ascending and sort_coords:
        da = da.sortby(lat_dim)

    # ---- reorder dims if possible
    dims = list(da.dims)
    wanted = [d for d in (time_dim, lat_dim, lon_dim) if d in dims]
    rest = [d for d in dims if d not in wanted]
    if wanted:
        da = da.transpose(*(wanted + rest))

    if cast_dtype is not None:
        da = da.astype(cast_dtype)

    return da

def _shift_time_coord(time_coord: xr.DataArray, hours: int) -> xr.DataArray:
    if hours == 0:
        return time_coord
    delta = _dt.timedelta(hours=int(hours))

    # cftime path (dtype=object)
    if time_coord.dtype == object:
        return xr.apply_ufunc(
            lambda x: x + delta,
            time_coord,
            vectorize=True,
            dask="parallelized",
            output_dtypes=[object],
        )

    # numpy datetime64 path
    return time_coord + np.timedelta64(int(hours), "h")

# ============================================================
# Loader helper (self-contained)
# ============================================================
def open_model_daily_series(
    base_dir: str,
    sub: str,
    var: str,
    time_slice: Tuple[str, str],
    *,
    time_dim: str = "time",
    lat_dim: str = "lat",
    lon_dim: str = "lon",
    time_shift_hours: int = -12,
    patterns: Optional[List[str]] = None,
) -> xr.DataArray:
    """
    Open model daily time series from files under base_dir/sub.
    Adjust patterns if needed for your naming convention.
    """
    start, end = time_slice
    d = Path(base_dir) / sub

    if patterns is None:
        patterns = [
            f"*{var}*{start[:4]}*.nc",
            f"*{var}*{end[:4]}*.nc",
            f"*{var}*.nc",
        ]

    files: List[str] = []
    for p in patterns:
        files.extend(glob.glob(str(d / p)))
    files = sorted(set(files))

    if not files:
        raise FileNotFoundError(f"No files under {d} for var={var}. Tried patterns={patterns}")
        
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    
    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        decode_times=time_coder,   # <-- instead of use_cftime=True
    )
    
    if var not in ds:
        raise KeyError(f"{var} not in {files[0]} vars={list(ds.data_vars)}")

    da = ds[var]
    da = da.assign_coords({time_dim: _shift_time_coord(da[time_dim], time_shift_hours)})

    # standardize coords now (so the rest of the pipeline is stable)
    da = standardize_spacetime(
        da,
        time_dim=time_dim, lat_dim=lat_dim, lon_dim=lon_dim,
        time_shift_hours=time_shift_hours,          # <-- KEY FIX for “end-of-day labeled” model output
        rename_latlon=True,
        floor_time_to_day=True,
        drop_duplicate_time=True,
        lon_to_0_360=True,
        drop_lon_360=True,
        sort_coords=True,
        ensure_lat_ascending=True,
    )
    
    t = da[time_dim].values
    print("dtype:", np.asarray(t).dtype)
    # show a small window around the problem
    sel = da.sel({time_dim: slice("2012-02-20", "2012-03-05")})
    print(sel[time_dim].values)

    diag_time_axis(da, time_dim, tag=f"LOADER STD {var}", drop_feb29_expected=True)
    
    # slice AFTER standardization (so slicing uses corrected time labels)
    da = da.sel({time_dim: slice(start, end)})
    
    diag_time_axis(da, time_dim, tag=f"LOADER STD+SLICE {var}", drop_feb29_expected=True)

    return da


# ============================================================
# Projector (self-contained)
# ============================================================
class RmmHindcastProjector:
    def __init__(self, cfg: RmmProjectConfig, *, verbose: bool = True, diag_level: int = 1, enforce_contract: bool = True):
        self.cfg = cfg
        self.verbose = bool(verbose)
        self.diag_level = int(diag_level)
        self.enforce_contract = bool(enforce_contract)

        self.ds_eof = xr.open_dataset(cfg.eof_nc)
        self._sync_from_eof_attrs()

    # -------------------------
    # logging
    # -------------------------
    def _log(self, msg: str) -> None:
        if self.verbose and self.diag_level > 0:
            print(f"[RMM-PROJECT] {msg}")

    def _log2(self, msg: str) -> None:
        if self.verbose and self.diag_level > 1:
            print(f"[RMM-PROJECT] {msg}")

    @staticmethod
    def print_field_summary(tag: str, da: xr.DataArray, *, time_dim="time", lat_dim="lat", lon_dim="lon") -> None:
        def _bounds(x):
            try:
                return f"{np.nanmin(x)} → {np.nanmax(x)}"
            except Exception:
                return "NA"

        print(f"[{tag}] dims={da.dims} shape={da.shape} dtype={da.dtype}")

        if time_dim in da.coords:
            t = da[time_dim].values
            print(f"[{tag}] time={_bounds(t)} (n={t.size})")
            if t.size > 1:
                td = t.astype("datetime64[D]").astype(int)
                dt = np.unique(np.diff(td))
                print(f"[{tag}] Δtime(unique days)={dt[:10]}{'...' if dt.size>10 else ''}")
                nunq = np.unique(td).size
                if nunq != td.size:
                    print(f"[{tag}] WARNING: duplicate daily timestamps: n={td.size} unique={nunq}")

        if lat_dim in da.coords:
            lat = da[lat_dim].values
            asc = bool(np.all(np.diff(lat) >= 0)) if lat.size > 1 else True
            print(f"[{tag}] lat={_bounds(lat)} (n={lat.size}) ascending={asc}")

        if lon_dim in da.coords:
            lon = da[lon_dim].values
            in0360 = bool((np.nanmin(lon) >= 0) and (np.nanmax(lon) < 360))
            print(f"[{tag}] lon={_bounds(lon)} (n={lon.size}) in_0_360={in0360}")

    # -------------------------
    # EOF attrs -> internal knobs
    # -------------------------
    def _sync_from_eof_attrs(self) -> None:
        cfg = self.cfg
        attrs = self.ds_eof.attrs

        def _as_int(x, default):
            try:
                return int(str(x))
            except Exception:
                return default

        def _as_bool(x, default):
            try:
                return str(x).lower() in ("true", "1", "yes")
            except Exception:
                return default

        self._flag_drop_leap_day = _as_bool(attrs.get("drop_leap_day", cfg.drop_leap_day), cfg.drop_leap_day)
        self._doy_smooth_days = _as_int(attrs.get("doy_smooth_days", cfg.doy_smooth_days), cfg.doy_smooth_days)

        self._train_start = str(attrs.get("train_start", cfg.climo_train_start))
        self._train_end = str(attrs.get("train_end", cfg.climo_train_end))

        # quick EOF sanity log
        self._log(f"EOF loaded: {Path(cfg.eof_nc).name}")
        self._log(f"Climo train window: {self._train_start} → {self._train_end}")
        for need in ["EOF_OLR", "EOF_U850", "EOF_U200", "pc_std_mode", "rmm_scale", "norm_std_olr", "norm_std_u850", "norm_std_u200"]:
            if need not in self.ds_eof:
                raise KeyError(f"EOF file missing required variable: {need}")

    # -------------------------
    # domain utilities
    # -------------------------
    def _ensure_lon_0_360(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
        lon = da[cfg.lon_dim]
        if np.nanmin(lon.values) < 0:
            da = da.assign_coords({cfg.lon_dim: (lon % 360).astype(lon.dtype)}).sortby(cfg.lon_dim)
        return da

    def _subset_domain(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
        da = self._ensure_lon_0_360(da)

        lat0, lat1 = cfg.lat_range
        lon0, lon1 = cfg.lon_range

        lat = da[cfg.lat_dim]
        if lat.size > 1 and lat[0] < lat[-1]:
            da = da.sel({cfg.lat_dim: slice(lat0, lat1)})
        else:
            da = da.sel({cfg.lat_dim: slice(lat1, lat0)})

        lon = da[cfg.lon_dim]
        if lon1 >= 360.0:
            m = (lon >= lon0) & (lon < 360.0)
            da = da.sel({cfg.lon_dim: lon.where(m, drop=True)})
        else:
            da = da.sel({cfg.lon_dim: slice(lon0, lon1)})

        return da

    def _drop_leap_day(self, da: xr.DataArray, time_dim: str) -> xr.DataArray:
        if not self._flag_drop_leap_day:
            return da
        t = da[time_dim]
        mask = ~((t.dt.month == 2) & (t.dt.day == 29))
        return da.sel({time_dim: mask})

    def _add_doy365_coord(self, da: xr.DataArray, time_dim: str) -> xr.DataArray:
        t = da[time_dim]
        doy = t.dt.dayofyear
        is_leap = t.dt.is_leap_year
        doy365 = xr.where(is_leap & (doy > 59), doy - 1, doy)
        return da.assign_coords(doy365=doy365)

    def _smooth_doy_climo_365(self, climo: xr.DataArray, win: int) -> xr.DataArray:
        if win is None or win <= 1:
            return climo

        doy = climo["doy365"]
        n = int(doy.max())
        pad = win // 2

        left = climo.sel(doy365=slice(n - pad + 1, n))
        right = climo.sel(doy365=slice(1, pad))

        padded = xr.concat([left, climo, right], dim="doy365")
        padded = padded.assign_coords(doy365=np.arange(1, padded.sizes["doy365"] + 1))

        sm = padded.rolling(doy365=win, center=True, min_periods=win).mean()
        sm = sm.isel(doy365=slice(pad, pad + n))
        sm = sm.assign_coords(doy365=np.arange(1, n + 1))
        return sm

    def _lat_weight(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
        lat = da[cfg.lat_dim]
        w = np.sqrt(np.cos(np.deg2rad(lat)))
        return da * w

    def _apply_obs_norm(self, da: xr.DataArray, which: str) -> xr.DataArray:
        ds = self.ds_eof
        if which == "olr":
            s = float(ds["norm_std_olr"].values)
        elif which == "u850":
            s = float(ds["norm_std_u850"].values)
        elif which == "u200":
            s = float(ds["norm_std_u200"].values)
        else:
            raise ValueError(which)
        if not np.isfinite(s) or s == 0.0:
            raise ValueError(f"Bad norm scalar for {which}: {s}")
        return da / s

    # -------------------------
    # obs climo builder (cached)
    # -------------------------
    def _climo_cache_paths(self) -> Dict[str, Path]:
        cfg = self.cfg
        d = Path(cfg.cache_dir)
        d.mkdir(parents=True, exist_ok=True)
        tag = (
            f"climo_{self._train_start}_{self._train_end}"
            f"_lat{cfg.lat_range[0]:g}_{cfg.lat_range[1]:g}"
            f"_lon{cfg.lon_range[0]:g}_{cfg.lon_range[1]:g}"
            f"_sm{self._doy_smooth_days}"
        )
        return {
            "olr": d / f"{tag}_OLR.nc",
            "u850": d / f"{tag}_U850.nc",
            "u200": d / f"{tag}_U200.nc",
        }

    def _open_obs_var(self, path: str, var: str) -> xr.DataArray:
        cfg = self.cfg
        ds = xr.open_dataset(path)
        if var not in ds:
            raise KeyError(f"Variable '{var}' not found in {path}. Vars: {list(ds.data_vars)}")
        da = ds[var]

        # rename
        rename_map = {}
        if "latitude" in da.dims and cfg.lat_dim not in da.dims:
            rename_map["latitude"] = cfg.lat_dim
        if "longitude" in da.dims and cfg.lon_dim not in da.dims:
            rename_map["longitude"] = cfg.lon_dim
        if rename_map:
            da = da.rename(rename_map)

        # singleton level
        if "level" in da.dims and da.sizes["level"] == 1:
            da = da.isel(level=0, drop=True)
        if "plev" in da.dims and da.sizes["plev"] == 1:
            da = da.isel(plev=0, drop=True)

        # standardize time to daily and drop dups
        da = standardize_spacetime(
            da,
            time_dim=cfg.time_dim, 
            time_shift_hours=cfg.obs_time_shift_hours,  # no shift for observation
            lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim,
            rename_latlon=False,  # already handled
            floor_time_to_day=True, drop_duplicate_time=True,
            lon_to_0_360=True, drop_lon_360=True,
            sort_coords=True, ensure_lat_ascending=True,
        )

        # enforce expected order
        da = da.transpose(cfg.time_dim, cfg.lat_dim, cfg.lon_dim)
        return da

    def build_obs_climatology(self) -> Dict[str, xr.DataArray]:
        cfg = self.cfg
        paths = self._climo_cache_paths()

        if cfg.use_cache and all(p.exists() for p in paths.values()) and (not cfg.overwrite_cache):
            self._log("Loading cached observed DOY climatology...")
            out = {
                "olr": xr.open_dataset(paths["olr"])["climo"].load(),
                "u850": xr.open_dataset(paths["u850"])["climo"].load(),
                "u200": xr.open_dataset(paths["u200"])["climo"].load(),
            }
            return out

        self._log("Building observed DOY climatology (slow once; then cached)...")

        lat_e = self.ds_eof[cfg.lat_dim]
        lon_e = self.ds_eof[cfg.lon_dim]

        def _make_one(path, var) -> xr.DataArray:
            da = self._open_obs_var(path, var)
            da = self._subset_domain(da)
            da = da.sel({cfg.time_dim: slice(self._train_start, self._train_end)}).sortby(cfg.time_dim)
            da = self._drop_leap_day(da, cfg.time_dim)
            da = self._add_doy365_coord(da, cfg.time_dim)

            # regrid to EOF grid
            if (da[cfg.lat_dim].size != lat_e.size) or (da[cfg.lon_dim].size != lon_e.size) or \
               (not np.allclose(da[cfg.lat_dim].values, lat_e.values)) or (not np.allclose(da[cfg.lon_dim].values, lon_e.values)):
                da = da.interp({cfg.lat_dim: lat_e, cfg.lon_dim: lon_e}, method=cfg.interp_method)

            climo = da.groupby("doy365").mean(cfg.time_dim)
            if self._doy_smooth_days and self._doy_smooth_days > 1:
                climo = self._smooth_doy_climo_365(climo, self._doy_smooth_days)
            return climo.astype("float32")

        cl_olr = _make_one(cfg.noaa_olr_file, cfg.v_olr)
        cl_u850 = _make_one(cfg.era5_u850_file, cfg.v_u850)
        cl_u200 = _make_one(cfg.era5_u200_file, cfg.v_u200)

        xr.Dataset({"climo": cl_olr}).to_netcdf(paths["olr"])
        xr.Dataset({"climo": cl_u850}).to_netcdf(paths["u850"])
        xr.Dataset({"climo": cl_u200}).to_netcdf(paths["u200"])

        return {"olr": cl_olr, "u850": cl_u850, "u200": cl_u200}
        
    def _assert_daily_time(self, da: xr.DataArray, tag: str):
        t = da[self.cfg.time_dim].values.astype("datetime64[D]")
        if np.unique(t).size != t.size:
            raise ValueError(f"{tag}: duplicate daily timestamps (need unique daily time).")
            
    def _assert_time_sorted(self, da: xr.DataArray, tag: str):
        t = da[self.cfg.time_dim].values
        try:
            t64 = t.astype("datetime64[ns]")
            if t64.size > 1 and np.any(np.diff(t64.astype("int64")) < 0):
                raise ValueError(f"{tag}: time not sorted ascending (expect standardized input).")
        except Exception:
            # fallback: compare as Python objects (works for cftime)
            if t.size > 1 and any(t[i] > t[i+1] for i in range(t.size - 1)):
                raise ValueError(f"{tag}: time not sorted ascending (non-numpy datetime).")

    # -------------------------
    # projection core
    # -------------------------
    def _project_X_to_rmm(self, X: np.ndarray) -> np.ndarray:
        ds = self.ds_eof

        eof1 = np.concatenate([
            ds.EOF_OLR.sel(mode=1).values.ravel(),
            ds.EOF_U850.sel(mode=1).values.ravel(),
            ds.EOF_U200.sel(mode=1).values.ravel(),
        ]).astype("float64", copy=False)

        eof2 = np.concatenate([
            ds.EOF_OLR.sel(mode=2).values.ravel(),
            ds.EOF_U850.sel(mode=2).values.ravel(),
            ds.EOF_U200.sel(mode=2).values.ravel(),
        ]).astype("float64", copy=False)

        pc1_raw = X @ eof1
        pc2_raw = X @ eof2

        pc_std = ds["pc_std_mode"].values.astype("float64")
        rmm_scale = float(ds["rmm_scale"].values)

        pc1 = (pc1_raw / pc_std[0]) / rmm_scale
        pc2 = (pc2_raw / pc_std[1]) / rmm_scale
        return np.stack([pc1, pc2], axis=1)
        
    def project_obs_to_rmm(
        self,
        start: str,
        end: str,
        *,
        climo: Optional[Dict[str, xr.DataArray]] = None,
        source_tag: str = "OBS",
        align_to_time: Optional[np.ndarray] = None,  # e.g., model time array; if None keep full obs window
        diag: bool = True,
    ) -> xr.Dataset:
        """
        Project observations (NOAA OLR + ERA5 U850/U200) to RMM using the SAME pipeline as model:
          standardize -> subset domain -> interp to EOF grid -> (obs - DOY climo) -> lat-weight -> obs norm -> EOF projection
    
        Output:
          Dataset(time) with RMM1_obs, RMM2_obs, RMM_amp_obs, valid_time
        """
        cfg = self.cfg
        if climo is None:
            climo = self.build_obs_climatology()
    
        self._log(f"=== {source_tag} projection ===")
        self._log(f"Window: {start} → {end}")
    
        # 1) Open obs fields (already standardized to daily + lon/lat conventions in _open_obs_var)
        olr = self._open_obs_var(cfg.noaa_olr_file, cfg.v_olr)
        u850 = self._open_obs_var(cfg.era5_u850_file, cfg.v_u850)
        u200 = self._open_obs_var(cfg.era5_u200_file, cfg.v_u200)
        
        # 2) Trim to requested window
        olr = olr.sel({cfg.time_dim: slice(start, end)})
        u850 = u850.sel({cfg.time_dim: slice(start, end)})
        u200 = u200.sel({cfg.time_dim: slice(start, end)})
    
        if diag and self.diag_level > 0:
            self.print_field_summary(f"{source_tag} RAW OLR", olr, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
            self.print_field_summary(f"{source_tag} RAW U850", u850, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
            self.print_field_summary(f"{source_tag} RAW U200", u200, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
    
        # 3) Subset domain + regrid to EOF grid
        lat_e = self.ds_eof[cfg.lat_dim]
        lon_e = self.ds_eof[cfg.lon_dim]
    
        def _prep(da: xr.DataArray) -> xr.DataArray:
            da = self._subset_domain(da)
            # interp to EOF lat/lon if needed
            if (da[cfg.lat_dim].size != lat_e.size) or (da[cfg.lon_dim].size != lon_e.size) or \
               (not np.allclose(da[cfg.lat_dim].values, lat_e.values)) or (not np.allclose(da[cfg.lon_dim].values, lon_e.values)):
                da = da.interp({cfg.lat_dim: lat_e, cfg.lon_dim: lon_e}, method=cfg.interp_method)
            return da
    
        olr = _prep(olr)
        u850 = _prep(u850)
        u200 = _prep(u200)
    
        # 4) Align time across the three obs fields (intersection)
        t = np.intersect1d(olr[cfg.time_dim].values, u850[cfg.time_dim].values)
        t = np.intersect1d(t, u200[cfg.time_dim].values)
        if t.size < 30:
            raise ValueError(f"{source_tag}: Too few common days after alignment: {t.size}")
    
        olr = olr.sel({cfg.time_dim: t})
        u850 = u850.sel({cfg.time_dim: t})
        u200 = u200.sel({cfg.time_dim: t})
    
        # Optional: align obs to a target time array (e.g. model time).
        # This will DROP obs days not present in align_to_time.
        if align_to_time is not None:
            tt = np.intersect1d(olr[cfg.time_dim].values, np.asarray(align_to_time))
            olr = olr.sel({cfg.time_dim: tt})
            u850 = u850.sel({cfg.time_dim: tt})
            u200 = u200.sel({cfg.time_dim: tt})
            if tt.size < 30:
                raise ValueError(f"{source_tag}: Too few days after align_to_time intersection: {tt.size}")
    
        # 5) Drop leap day (to match climo contract)
        olr = self._drop_leap_day(olr, cfg.time_dim)
        u850 = self._drop_leap_day(u850, cfg.time_dim)
        u200 = self._drop_leap_day(u200, cfg.time_dim)
    
        # 6) Anomalies vs obs DOY climatology (same as model)
        def _anom(da: xr.DataArray, cl: xr.DataArray) -> xr.DataArray:
            da = self._add_doy365_coord(da, cfg.time_dim)
            return da.groupby("doy365") - cl
    
        olr_a = _anom(olr, climo["olr"])
        u850_a = _anom(u850, climo["u850"])
        u200_a = _anom(u200, climo["u200"])
    
        # 7) Lat-weight + obs norm scalars
        olr_p = self._apply_obs_norm(self._lat_weight(olr_a), "olr")
        u850_p = self._apply_obs_norm(self._lat_weight(u850_a), "u850")
        u200_p = self._apply_obs_norm(self._lat_weight(u200_a), "u200")
    
        # 8) Conservative NaN drop (same logic as model)
        ok = (
            np.isfinite(olr_p).all(dim=(cfg.lat_dim, cfg.lon_dim)) &
            np.isfinite(u850_p).all(dim=(cfg.lat_dim, cfg.lon_dim)) &
            np.isfinite(u200_p).all(dim=(cfg.lat_dim, cfg.lon_dim))
        )
        if hasattr(ok.data, "__dask_graph__"):
            ok = ok.compute()
    
        olr_p = olr_p.where(ok, drop=True)
        u850_p = u850_p.where(ok, drop=True)
        u200_p = u200_p.where(ok, drop=True)
    
        # 9) Stack -> project
        olr_s = olr_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))
        u850_s = u850_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))
        u200_s = u200_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))
    
        X = np.concatenate([olr_s.values, u850_s.values, u200_s.values], axis=-1).astype("float64", copy=False)
        n_time = olr_s.sizes[cfg.time_dim]
        pcs = self._project_X_to_rmm(X.reshape(n_time, -1))
    
        rmm1 = xr.DataArray(
            pcs[:, 0].astype("float32"),
            dims=(cfg.time_dim,),
            coords={cfg.time_dim: olr_s[cfg.time_dim]},
            name="RMM1_obs",
        )
        rmm2 = xr.DataArray(
            pcs[:, 1].astype("float32"),
            dims=(cfg.time_dim,),
            coords={cfg.time_dim: olr_s[cfg.time_dim]},
            name="RMM2_obs",
        )
        amp = xr.apply_ufunc(lambda a, b: np.sqrt(a*a + b*b), rmm1, rmm2).rename("RMM_amp_obs")
        vt = rmm1[cfg.time_dim].rename("valid_time")
    
        out = xr.Dataset({"RMM1_obs": rmm1, "RMM2_obs": rmm2, "RMM_amp_obs": amp, "valid_time": vt})
    
        if diag:
            try:
                self._log(
                    f"{source_tag} projected: ntime={out[cfg.time_dim].size} "
                    f"RMM1 std={float(out['RMM1_obs'].std()):.3f} "
                    f"RMM2 std={float(out['RMM2_obs'].std()):.3f} "
                    f"AMP mean={float(out['RMM_amp_obs'].mean()):.3f}"
                )
            except Exception:
                pass
    
            # Missing-day report (optional but very helpful)
            try:
                report_missing_days(out[cfg.time_dim].values, tag=f"{source_tag} RMM")
            except Exception:
                pass
    
        return out

        
    def project_model_to_rmm(
        self,
        fc_olr: xr.DataArray,
        fc_u850: xr.DataArray,
        fc_u200: xr.DataArray,
        *,
        climo: Optional[Dict[str, xr.DataArray]] = None,
        exp_tag: Optional[str] = None,
    ) -> xr.Dataset:
        """
        Input: model daily diagnostics on (time,lat,lon), or (init,time,lat,lon) with init size 1.
        Output: Dataset with RMM1_fcst/RMM2_fcst/RMM_amp_fcst and valid_time coord.
        """
        cfg = self.cfg
        if exp_tag:
            self._log(f"=== {exp_tag} ===")

        climo = climo or self.build_obs_climatology()

        # If init dim exists, expect single init for this pipeline
        if cfg.init_dim in fc_olr.dims:
            if fc_olr.sizes[cfg.init_dim] != 1:
                raise ValueError("This projector currently supports single-init only.")
            init_val = fc_olr[cfg.init_dim].values[0]
            fc_olr0 = fc_olr.isel({cfg.init_dim: 0})
            fc_u8500 = fc_u850.isel({cfg.init_dim: 0})
            fc_u2000 = fc_u200.isel({cfg.init_dim: 0})
        else:
            init_val = None
            fc_olr0, fc_u8500, fc_u2000 = fc_olr, fc_u850, fc_u200

        # Diagnostics after standardization
        if self.diag_level > 1:
            self.print_field_summary(f"{exp_tag or 'MODEL'} STD OLR", fc_olr0, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
            self.print_field_summary(f"{exp_tag or 'MODEL'} STD U850", fc_u8500, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
            self.print_field_summary(f"{exp_tag or 'MODEL'} STD U200", fc_u2000, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)

        # Trim verify window if requested
        if cfg.verify_start or cfg.verify_end:
            s = cfg.verify_start or "0001-01-01"
            e = cfg.verify_end or "9999-12-31"
            fc_olr0 = fc_olr0.sel({cfg.time_dim: slice(s, e)})
            fc_u8500 = fc_u8500.sel({cfg.time_dim: slice(s, e)})
            fc_u2000 = fc_u2000.sel({cfg.time_dim: slice(s, e)})

        # Subset + regrid to EOF grid
        lat_e = self.ds_eof[cfg.lat_dim]
        lon_e = self.ds_eof[cfg.lon_dim]

        def _prep(da: xr.DataArray) -> xr.DataArray:
            da = self._subset_domain(da)
            if (da[cfg.lat_dim].size != lat_e.size) or (da[cfg.lon_dim].size != lon_e.size) or \
               (not np.allclose(da[cfg.lat_dim].values, lat_e.values)) or (not np.allclose(da[cfg.lon_dim].values, lon_e.values)):
                da = da.interp({cfg.lat_dim: lat_e, cfg.lon_dim: lon_e}, method=cfg.interp_method)
            return da

        fc_olr0 = _prep(fc_olr0)
        fc_u8500 = _prep(fc_u8500)
        fc_u2000 = _prep(fc_u2000)

        # Align time across three fields
        t = np.intersect1d(fc_olr0[cfg.time_dim].values, fc_u8500[cfg.time_dim].values)
        t = np.intersect1d(t, fc_u2000[cfg.time_dim].values)
        if t.size < 30:
            raise ValueError(f"Too few common model days after alignment: {t.size}")
        fc_olr0 = fc_olr0.sel({cfg.time_dim: t})
        fc_u8500 = fc_u8500.sel({cfg.time_dim: t})
        fc_u2000 = fc_u2000.sel({cfg.time_dim: t})
        
        # Drop leap day
        fc_olr0 = self._drop_leap_day(fc_olr0, cfg.time_dim)
        fc_u8500 = self._drop_leap_day(fc_u8500, cfg.time_dim)
        fc_u2000 = self._drop_leap_day(fc_u2000, cfg.time_dim)

        if self.enforce_contract:
            self._assert_daily_time(fc_olr0, f"{exp_tag} OLR")
            self._assert_daily_time(fc_u8500, f"{exp_tag} U850")
            self._assert_daily_time(fc_u2000, f"{exp_tag} U200")
            self._assert_time_sorted(fc_olr0, f"{exp_tag} OLR")
            self._assert_time_sorted(fc_u8500, f"{exp_tag} U850")
            self._assert_time_sorted(fc_u2000, f"{exp_tag} U200")

        # Anomalies vs obs DOY climo
        def _anom(da: xr.DataArray, cl: xr.DataArray) -> xr.DataArray:
            da = self._add_doy365_coord(da, cfg.time_dim)
            return da.groupby("doy365") - cl

        olr_a = _anom(fc_olr0, climo["olr"])
        u850_a = _anom(fc_u8500, climo["u850"])
        u200_a = _anom(fc_u2000, climo["u200"])

        # Lat weight + obs norm scalars
        olr_p = self._apply_obs_norm(self._lat_weight(olr_a), "olr")
        u850_p = self._apply_obs_norm(self._lat_weight(u850_a), "u850")
        u200_p = self._apply_obs_norm(self._lat_weight(u200_a), "u200")

        # Drop any times with NaNs across vars (conservative)
        ok = (
            np.isfinite(olr_p).all(dim=(cfg.lat_dim, cfg.lon_dim)) &
            np.isfinite(u850_p).all(dim=(cfg.lat_dim, cfg.lon_dim)) &
            np.isfinite(u200_p).all(dim=(cfg.lat_dim, cfg.lon_dim))
        )
        if hasattr(ok.data, "__dask_graph__"):
            ok = ok.compute()

        olr_p = olr_p.where(ok, drop=True)
        u850_p = u850_p.where(ok, drop=True)
        u200_p = u200_p.where(ok, drop=True)

        # Stack -> project
        olr_s = olr_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))
        u850_s = u850_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))
        u200_s = u200_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))

        X = np.concatenate([olr_s.values, u850_s.values, u200_s.values], axis=-1).astype("float64", copy=False)
        n_time = olr_s.sizes[cfg.time_dim]
        pcs = self._project_X_to_rmm(X.reshape(n_time, -1))

        rmm1 = xr.DataArray(pcs[:, 0].astype("float32"), dims=(cfg.time_dim,), coords={cfg.time_dim: olr_s[cfg.time_dim]}, name="RMM1_fcst")
        rmm2 = xr.DataArray(pcs[:, 1].astype("float32"), dims=(cfg.time_dim,), coords={cfg.time_dim: olr_s[cfg.time_dim]}, name="RMM2_fcst")
        amp = xr.apply_ufunc(lambda a, b: np.sqrt(a*a + b*b), rmm1, rmm2).rename("RMM_amp_fcst")

        # Wrap back to (init,time) if we had init
        if init_val is not None:
            rmm1 = rmm1.expand_dims({cfg.init_dim: [np.datetime64(init_val).astype("datetime64[ns]")]})
            rmm2 = rmm2.expand_dims({cfg.init_dim: [np.datetime64(init_val).astype("datetime64[ns]")]})
            amp = amp.expand_dims({cfg.init_dim: [np.datetime64(init_val).astype("datetime64[ns]")]})

            # valid_time as (init,time)
            vt = xr.DataArray(
                np.array(rmm1[cfg.time_dim].values, dtype="datetime64[ns]")[None, :],
                dims=(cfg.init_dim, cfg.time_dim),
                coords={cfg.init_dim: rmm1[cfg.init_dim], cfg.time_dim: rmm1[cfg.time_dim]},
                name="valid_time",
            )
        else:
            vt = rmm1[cfg.time_dim].rename("valid_time")

        out = xr.Dataset({"RMM1_fcst": rmm1, "RMM2_fcst": rmm2, "RMM_amp_fcst": amp, "valid_time": vt})

        # Summary diagnostics
        try:
            self._log(f"Projected: ntime={out[cfg.time_dim].size} "
                      f"RMM1 std={float(out['RMM1_fcst'].std()):.3f} "
                      f"RMM2 std={float(out['RMM2_fcst'].std()):.3f} "
                      f"AMP mean={float(out['RMM_amp_fcst'].mean()):.3f}")
        except Exception:
            pass

        return out

    def to_timeseries(self, fc_rmm: xr.Dataset) -> xr.Dataset:
        """(init,time) with init size=1 -> (time) with scalar init coord."""
        cfg = self.cfg
        if cfg.init_dim not in fc_rmm.dims or fc_rmm.sizes[cfg.init_dim] != 1:
            raise ValueError("to_timeseries requires init dim size=1")

        r1 = fc_rmm["RMM1_fcst"].isel({cfg.init_dim: 0})
        r2 = fc_rmm["RMM2_fcst"].isel({cfg.init_dim: 0})
        vt = fc_rmm["valid_time"].isel({cfg.init_dim: 0}) if (cfg.init_dim in fc_rmm["valid_time"].dims) else fc_rmm[cfg.time_dim]

        ds_ts = xr.Dataset(
            {"RMM1_fcst": (("time",), r1.values.astype("float32")),
             "RMM2_fcst": (("time",), r2.values.astype("float32")),
             "RMM_amp_fcst": (("time",), fc_rmm["RMM_amp_fcst"].isel({cfg.init_dim: 0}).values.astype("float32"))},
            coords={"time": vt.values.astype("datetime64[ns]")},
        ).sortby("time")

        init_val = np.datetime64(fc_rmm[cfg.init_dim].values[0]).astype("datetime64[ns]")
        ds_ts = ds_ts.assign_coords({cfg.init_dim: init_val})

        return ds_ts

    def project_and_save(
        self,
        fc_olr: xr.DataArray,
        fc_u850: xr.DataArray,
        fc_u200: xr.DataArray,
        out_nc: Path,
        *,
        climo: Optional[Dict[str, xr.DataArray]] = None,
        overwrite: bool = True,
        exp_tag: Optional[str] = None,
    ) -> RmmProjectionResult:
        if (not overwrite) and out_nc.exists():
            raise FileExistsError(str(out_nc))

        if climo is None:
            climo = self.build_obs_climatology()

        fc_rmm = self.project_model_to_rmm(
            fc_olr, fc_u850, fc_u200,
            climo=climo, exp_tag=exp_tag
        )

        out_nc.parent.mkdir(parents=True, exist_ok=True)
        self._log(f"Writing native -> {out_nc}")
        fc_rmm.to_netcdf(out_nc)
        
        # no timeseries output needed anymore
        return RmmProjectionResult(
            fc_rmm_native=fc_rmm,
            fc_rmm_timeseries=None
        )

In [3]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "mjo"
    out_dir.mkdir(parents=True, exist_ok=True)

    exp_list = ["CTRL", "DARTOP", "ERA5OP", "ERA5DL", "ERA5BL", "DARTBL", "DARTBA", "DARTAE"]

    period = "2012-2013"
    start, end = "2012-01-01", "2013-12-31"
    time_slice = (start, end)

    cfg = RmmProjectConfig(
        eof_nc=str(out_dir/"rmm_obs_eof_1deg_1986-2017_climo30_rm120.nc"),
        noaa_olr_file="/lcrc/soft/climate/e3sm_diags_data/obs_for_e3sm_diags/time-series/NOAA-OLR_Daily/FLUT_198601_202112.nc",
        era5_u850_file="/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/obs_data/ERA5/daily/U850_197901_201712.nc",
        era5_u200_file="/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/obs_data/ERA5/daily/U200_197901_201712.nc",
        verify_start=start,
        verify_end=end,
        lat_range=(-15.0, 15.0),
        lon_range=(0.0, 360.0),
        cache_dir=str(out_dir / "rmm_cache"),
        use_cache=True,
        overwrite_cache=False,
    )

    proj = RmmHindcastProjector(cfg, verbose=True, diag_level=2, enforce_contract=True)

    # Build climo once (cached)
    climo = proj.build_obs_climatology()
    print("[MAIN] Climatology ready:")
    for k in ("olr", "u850", "u200"):
        cl = climo[k]
        doymin = int(cl["doy365"].values.min()) if "doy365" in cl.coords else -1
        doymax = int(cl["doy365"].values.max()) if "doy365" in cl.coords else -1
        print(f"[MAIN]  {k}: dims={cl.dims} doy=[{doymin}-{doymax}] lat={cl.sizes[cfg.lat_dim]} lon={cl.sizes[cfg.lon_dim]}")

    # process observation : read -> summaries -> project/save
    obs_nc = out_dir / f"rmm_obs_projected_{period}.nc"
    if not obs_nc.exists():
        obs_rmm = proj.project_obs_to_rmm(
            start, end,
            climo=climo,
            source_tag="OBS",
            align_to_time=None,   # <-- project full obs window
            diag=True,
        )
        obs_rmm.to_netcdf(obs_nc)
        print(f"[MAIN] Saved OBS projection: {obs_nc}")
    else:
        print(f"[MAIN] Using existing OBS projection: {obs_nc}")
    
    obs_rmm = xr.open_dataset(obs_nc)
    print(f"[OBS] ntime={obs_rmm.sizes['time']} time={obs_rmm['time'].values[0]} → {obs_rmm['time'].values[-1]}")
    print(f"[OBS] std: RMM1={float(obs_rmm['RMM1_obs'].std()):.3f} RMM2={float(obs_rmm['RMM2_obs'].std()):.3f}")
    report_missing_days(obs_rmm["time"].values, tag="OBS")

    # Loop experiments: read -> summaries -> project/save
    for exp in exp_list:
        print(f"\n[MAIN] ===== {exp} =====")
        run_name = run_dict[exp]["name"]
        run_base_dir = f"/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings/{run_name}"
        sub = "post/atm/180x360_aave/ts/daily/1yr"

        # Read model fields (daily)
        fc_olr = open_model_daily_series(run_base_dir, sub, "FLUT", time_slice, 
                                         time_shift_hours=cfg.model_time_shift_hours,
                                         time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
        fc_u850 = open_model_daily_series(run_base_dir, sub, "U850", time_slice,
                                          time_shift_hours=cfg.model_time_shift_hours,
                                          time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
        fc_u200 = open_model_daily_series(run_base_dir, sub, "U200", time_slice,
                                          time_shift_hours=cfg.model_time_shift_hours,
                                          time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)

        # RAW summaries
        proj.print_field_summary(f"{exp} RAW OLR", fc_olr, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
        proj.print_field_summary(f"{exp} RAW U850", fc_u850, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)
        proj.print_field_summary(f"{exp} RAW U200", fc_u200, time_dim=cfg.time_dim, lat_dim=cfg.lat_dim, lon_dim=cfg.lon_dim)

        # Single-init convention
        init_time = np.datetime64(start)
        fc_olr = fc_olr.expand_dims(init=[init_time])
        fc_u850 = fc_u850.expand_dims(init=[init_time])
        fc_u200 = fc_u200.expand_dims(init=[init_time])

        # Project + save
        out_nc = out_dir / f"rmm_fcst_projected_{exp}_{period}.nc"
        res = proj.project_and_save(
            fc_olr=fc_olr, fc_u850=fc_u850, fc_u200=fc_u200,
            out_nc=out_nc,
            climo=climo,
            overwrite=True,
            exp_tag=exp,
        )

        # Post-I/O quick check (timeseries)
        ds_ts = xr.open_dataset(out_nc)
        print(f"[MAIN] TS check: ntime={ds_ts.sizes['time']} time={ds_ts['time'].values[0]} → {ds_ts['time'].values[-1]}")
        print(f"[MAIN] TS std: RMM1={float(ds_ts['RMM1_fcst'].std()):.3f} RMM2={float(ds_ts['RMM2_fcst'].std()):.3f}")
        report_missing_days(ds_ts["time"].values, tag=f"{exp} MODEL TS")


[RMM-PROJECT] EOF loaded: rmm_obs_eof_1deg_1986-2017_climo30_rm120.nc
[RMM-PROJECT] Climo train window: 1986-01-01 → 2017-12-31
[RMM-PROJECT] Loading cached observed DOY climatology...
[MAIN] Climatology ready:
[MAIN]  olr: dims=('doy365', 'lat', 'lon') doy=[1-365] lat=30 lon=360
[MAIN]  u850: dims=('doy365', 'lat', 'lon') doy=[1-365] lat=30 lon=360
[MAIN]  u200: dims=('doy365', 'lat', 'lon') doy=[1-365] lat=30 lon=360
[RMM-PROJECT] === OBS projection ===
[RMM-PROJECT] Window: 2012-01-01 → 2013-12-31
[OBS RAW OLR] dims=('time', 'lat', 'lon') shape=(731, 180, 360) dtype=float32
[OBS RAW OLR] time=2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000 (n=731)
[OBS RAW OLR] Δtime(unique days)=[1]
[OBS RAW OLR] lat=-89.5 → 89.5 (n=180) ascending=True
[OBS RAW OLR] lon=0.5 → 359.5 (n=360) in_0_360=True
[OBS RAW U850] dims=('time', 'lat', 'lon') shape=(731, 180, 360) dtype=float32
[OBS RAW U850] time=2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000 (n=731)
[OBS RAW U850] 

/tmp/ipykernel_2584203/3312015856.py:307: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


dtype: datetime64[ns]
['2012-02-20T00:00:00.000000000' '2012-02-21T00:00:00.000000000'
 '2012-02-22T00:00:00.000000000' '2012-02-23T00:00:00.000000000'
 '2012-02-24T00:00:00.000000000' '2012-02-25T00:00:00.000000000'
 '2012-02-26T00:00:00.000000000' '2012-02-27T00:00:00.000000000'
 '2012-02-28T00:00:00.000000000' '2012-03-01T00:00:00.000000000'
 '2012-03-02T00:00:00.000000000' '2012-03-03T00:00:00.000000000'
 '2012-03-04T00:00:00.000000000' '2012-03-05T00:00:00.000000000']
[LOADER STD FLUT] time(ns): 2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000  n=730
[LOADER STD FLUT] time(D):  2012-01-01 → 2013-12-31  n=730  unique_days=730
[LOADER STD FLUT] Δday(unique)=[1 2]
[LOADER STD FLUT] Missing days: 0
[LOADER STD+SLICE FLUT] time(ns): 2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000  n=730
[LOADER STD+SLICE FLUT] time(D):  2012-01-01 → 2013-12-31  n=730  unique_days=730
[LOADER STD+SLICE FLUT] Δday(unique)=[1 2]
[LOADER STD+SLICE FLUT] Missing days: 0


/tmp/ipykernel_2584203/3312015856.py:307: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


dtype: datetime64[ns]
['2012-02-20T00:00:00.000000000' '2012-02-21T00:00:00.000000000'
 '2012-02-22T00:00:00.000000000' '2012-02-23T00:00:00.000000000'
 '2012-02-24T00:00:00.000000000' '2012-02-25T00:00:00.000000000'
 '2012-02-26T00:00:00.000000000' '2012-02-27T00:00:00.000000000'
 '2012-02-28T00:00:00.000000000' '2012-03-01T00:00:00.000000000'
 '2012-03-02T00:00:00.000000000' '2012-03-03T00:00:00.000000000'
 '2012-03-04T00:00:00.000000000' '2012-03-05T00:00:00.000000000']
[LOADER STD U850] time(ns): 2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000  n=730
[LOADER STD U850] time(D):  2012-01-01 → 2013-12-31  n=730  unique_days=730
[LOADER STD U850] Δday(unique)=[1 2]
[LOADER STD U850] Missing days: 0
[LOADER STD+SLICE U850] time(ns): 2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000  n=730
[LOADER STD+SLICE U850] time(D):  2012-01-01 → 2013-12-31  n=730  unique_days=730
[LOADER STD+SLICE U850] Δday(unique)=[1 2]
[LOADER STD+SLICE U850] Missing days: 0


...

dtype: datetime64[ns]
['2012-02-20T00:00:00.000000000' '2012-02-21T00:00:00.000000000'
 '2012-02-22T00:00:00.000000000' '2012-02-23T00:00:00.000000000'
 '2012-02-24T00:00:00.000000000' '2012-02-25T00:00:00.000000000'
 '2012-02-26T00:00:00.000000000' '2012-02-27T00:00:00.000000000'
 '2012-02-28T00:00:00.000000000' '2012-03-01T00:00:00.000000000'
 '2012-03-02T00:00:00.000000000' '2012-03-03T00:00:00.000000000'
 '2012-03-04T00:00:00.000000000' '2012-03-05T00:00:00.000000000']
[LOADER STD FLUT] time(ns): 2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000  n=702
[LOADER STD FLUT] time(D):  2012-01-01 → 2013-12-31  n=702  unique_days=702
[LOADER STD FLUT] Δday(unique)=[ 1  2 29]
[LOADER STD FLUT] MISSING days: 28 (first 20) ['2012-07-01' '2012-07-02' '2012-07-03' '2012-07-04' '2012-07-05'
 '2012-07-06' '2012-07-07' '2012-07-08' '2012-07-09' '2012-07-10'
 '2012-07-11' '2012-07-12' '2012-07-13' '2012-07-14' '2012-07-15'
 '2012-07-16' '2012-07-17' '2012-07-18' '2012-07-19' '2012-07-2

/tmp/ipykernel_2584203/3312015856.py:307: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


dtype: datetime64[ns]
['2012-02-20T00:00:00.000000000' '2012-02-21T00:00:00.000000000'
 '2012-02-22T00:00:00.000000000' '2012-02-23T00:00:00.000000000'
 '2012-02-24T00:00:00.000000000' '2012-02-25T00:00:00.000000000'
 '2012-02-26T00:00:00.000000000' '2012-02-27T00:00:00.000000000'
 '2012-02-28T00:00:00.000000000' '2012-03-01T00:00:00.000000000'
 '2012-03-02T00:00:00.000000000' '2012-03-03T00:00:00.000000000'
 '2012-03-04T00:00:00.000000000' '2012-03-05T00:00:00.000000000']
[LOADER STD U850] time(ns): 2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000  n=702
[LOADER STD U850] time(D):  2012-01-01 → 2013-12-31  n=702  unique_days=702
[LOADER STD U850] Δday(unique)=[ 1  2 29]
[LOADER STD U850] MISSING days: 28 (first 20) ['2012-07-01' '2012-07-02' '2012-07-03' '2012-07-04' '2012-07-05'
 '2012-07-06' '2012-07-07' '2012-07-08' '2012-07-09' '2012-07-10'
 '2012-07-11' '2012-07-12' '2012-07-13' '2012-07-14' '2012-07-15'
 '2012-07-16' '2012-07-17' '2012-07-18' '2012-07-19' '2012-07-2

/tmp/ipykernel_2584203/3312015856.py:307: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


dtype: datetime64[ns]
['2012-02-20T00:00:00.000000000' '2012-02-21T00:00:00.000000000'
 '2012-02-22T00:00:00.000000000' '2012-02-23T00:00:00.000000000'
 '2012-02-24T00:00:00.000000000' '2012-02-25T00:00:00.000000000'
 '2012-02-26T00:00:00.000000000' '2012-02-27T00:00:00.000000000'
 '2012-02-28T00:00:00.000000000' '2012-03-01T00:00:00.000000000'
 '2012-03-02T00:00:00.000000000' '2012-03-03T00:00:00.000000000'
 '2012-03-04T00:00:00.000000000' '2012-03-05T00:00:00.000000000']
[LOADER STD U200] time(ns): 2012-01-01T00:00:00.000000000 → 2013-12-31T00:00:00.000000000  n=702
[LOADER STD U200] time(D):  2012-01-01 → 2013-12-31  n=702  unique_days=702
[LOADER STD U200] Δday(unique)=[ 1  2 29]
[LOADER STD U200] MISSING days: 28 (first 20) ['2012-07-01' '2012-07-02' '2012-07-03' '2012-07-04' '2012-07-05'
 '2012-07-06' '2012-07-07' '2012-07-08' '2012-07-09' '2012-07-10'
 '2012-07-11' '2012-07-12' '2012-07-13' '2012-07-14' '2012-07-15'
 '2012-07-16' '2012-07-17' '2012-07-18' '2012-07-19' '2012-07-2